In [353]:
import pandas as pd
import numpy as np

In [354]:
df = pd.read_json("../tf.json")
df.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-10-26,12422,8842,45914,0.00452,197.1,organic_search Variant 0001
1,2019-02-15,26260,19148,97929,0.00568,116.5,paid_search Variant 0002
2,2019-01-12,14383,11480,57737,0.00466,170.6,email_campaign Variant 0003
3,2017-09-30,22021,16969,95903,0.00344,111.8,paid_search Variant 0004
4,2021-08-12,38269,29771,208637,0.00442,210.7,paid_search Variant 0005


In [355]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      1000 non-null   str    
 1   sessions                  999 non-null    object 
 2   unique_visitors           1000 non-null   object 
 3   page_views                1000 non-null   int64  
 4   bounce_rate               1000 non-null   float64
 5   avg_session_duration_sec  1000 non-null   float64
 6   traffic_source            1000 non-null   str    
dtypes: float64(2), int64(1), object(2), str(2)
memory usage: 54.8+ KB


In [356]:
# date là khóa chính của bảng traffic (theo Dictionary): không được trùng
print('Số date trùng:', df["date"].duplicated().sum())

Số date trùng: 0


In [357]:
# Đếm các giá trị không đọc được của từng cột (null, chuỗi rỗng, chuỗi lỗi)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print('date lỗi:', df['date'].isna().sum())
df['sessions'] = pd.to_numeric(df["sessions"], errors="coerce")
print('sessions lỗi:', df['sessions'].isna().sum())
df['unique_visitors'] = pd.to_numeric(df["unique_visitors"], errors="coerce")
print('unique_visitors lỗi:', df['unique_visitors'].isna().sum())

date lỗi: 2
sessions lỗi: 2
unique_visitors lỗi: 1


In [358]:
sessions_notcorrect = df["sessions"] < 0
print('Số phiên âm (bất khả thi):', sessions_notcorrect.sum())
df.loc[sessions_notcorrect, "sessions"] = pd.NA
df["sessions"] = df["sessions"].fillna(df["sessions"].median())

Số phiên âm (bất khả thi): 1


In [359]:
display(df[df['date'].isna()])
display(df[df['sessions'].isna()])
display(df[df['unique_visitors'].isna()])


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
515,NaT,38996.0,31829.0,154237,0.00419,185.4,organic_search Variant 0516
728,NaT,16845.0,12989.0,54823,0.00338,269.9,email_campaign Variant 0729


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
633,2015-11-20,13750.0,NaN,61102,0.00364,313.1,paid_search Variant 0634


In [360]:
median_sessions = df["sessions"].median()
median_visitors = df["unique_visitors"].median()

df["sessions"] = df["sessions"].fillna(median_sessions)
df["unique_visitors"] = df["unique_visitors"].fillna(median_visitors)

df.dropna(subset=["date"], inplace=True)


In [361]:
# Chuẩn hóa khoảng trắng của traffic_source
print('Trước chuẩn hóa :', repr(df.loc[861, "traffic_source"]))
df["traffic_source"] = df["traffic_source"].str.strip().str.replace(r"\\s+", " ", regex=True)
print('Sau chuẩn hóa   :', repr(df.loc[861, "traffic_source"]))

Trước chuẩn hóa : '   paid_search Variant 0862   '
Sau chuẩn hóa   : 'paid_search Variant 0862'


In [362]:
df["traffic_source"].value_counts(dropna=False)

traffic_source
organic_search Variant 0001    1
paid_search Variant 0002       1
email_campaign Variant 0003    1
paid_search Variant 0004       1
paid_search Variant 0005       1
                              ..
organic_search Variant 0996    1
social_media Variant 0997      1
paid_search Variant 0998       1
email_campaign Variant 0999    1
organic_search Variant 1000    1
Name: count, Length: 998, dtype: int64

In [363]:
empty_source = df["traffic_source"].isin(["", "N/A", "n/a", None])
display(empty_source.sum())
df.loc[empty_source, "traffic_source"] = "Unknown"
df["traffic_source"] = df["traffic_source"].str.strip()



np.int64(2)

In [364]:
# Chuyển cột phân loại sang kiểu category để tiết kiệm bộ nhớ
df["traffic_source"] = df["traffic_source"].astype("category")
df.info()

<class 'pandas.DataFrame'>
Index: 998 entries, 0 to 999
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      998 non-null    datetime64[us]
 1   sessions                  998 non-null    float64       
 2   unique_visitors           998 non-null    float64       
 3   page_views                998 non-null    int64         
 4   bounce_rate               998 non-null    float64       
 5   avg_session_duration_sec  998 non-null    float64       
 6   traffic_source            998 non-null    category      
dtypes: category(1), datetime64[us](1), float64(4), int64(1)
memory usage: 96.6 KB


In [365]:
# Tổng kết trước khi xuất: không còn trùng, không còn kiểu dữ liệu sai
print('Tổng số date trùng:', df["date"].duplicated().sum())
print('Tổng số null      :', df.isna().sum().sum())
print()
print(df.isna().sum())


Tổng số date trùng: 0
Tổng số null      : 0

date                        0
sessions                    0
unique_visitors             0
page_views                  0
bounce_rate                 0
avg_session_duration_sec    0
traffic_source              0
dtype: int64


In [366]:
# Xuất dữ liệu Silver (đã chuẩn hóa kiểu)
df.to_csv("../SilverData/tf.csv", index=False)
print('Đã xuất', len(df), 'dòng ra ../SilverData/tf.csv')

Đã xuất 998 dòng ra ../SilverData/tf.csv
